# Serving Feature Generation — Hourly (Gold V2)

**Overview**
This notebook generates the feature set required for real-time or near-real-time predictions using the Gold V2 structure.

The features are constructed to match the schema used during model training.

**Objective**
Prepare model-ready features at station-hour level to enable prediction of demand and imbalance.

**Output**
- Feature table aligned with model input requirements

In [0]:
%pip install lightgbm
%restart_python

**Feature Construction**

This section builds temporal, lag-based, and contextual features using the latest available data.

The goal is to replicate the training feature logic to ensure consistency during prediction.

## JOB A - Build Serving Features Hourly (LightGBM Pipeline)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# JOB A - Build Serving Features Hourly (Incremental, idempotent)
# From GOLD_V2 -> SERVING_FEATURES_DIR/hour_key=YYYYMMDDHH
# Also writes LATEST.txt (pointer)
# Serverless-safe (Spark Connect compatible)
# Feature engineering is identical to XGB/RF versions
# ============================================================

GOLD_V2_DIR          = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/data/gold/gold_v2_spatiotemporal_events"
SERVING_FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
LATEST_PTR_PATH      = f"{SERVING_FEATURES_DIR}/LATEST.txt"
LOOKBACK_HOURS       = 220  # >= 168 + margin

USE_EXPLICIT_TARGET                    = False
TARGET_Y, TARGET_M, TARGET_D, TARGET_H = None, None, None, None

def path_exists(path: str) -> bool:
    try: dbutils.fs.ls(path); return True
    except: return False

def hour_key_from_ts(ts) -> str:
    return f"{ts.year:04d}{ts.month:02d}{ts.day:02d}{ts.hour:02d}"

if not path_exists(GOLD_V2_DIR):
    raise Exception(f"GOLD_V2_DIR not found: {GOLD_V2_DIR}")

# Read + create ts_hour
df = spark.read.parquet(GOLD_V2_DIR)
df = df.withColumn(
    "ts_hour",
    F.to_timestamp(F.concat_ws(" ", F.make_date("year","month","day"), F.format_string("%02d:00:00", F.col("hour"))))
)

if USE_EXPLICIT_TARGET:
    target_ts_val = spark.range(1).select(
        F.to_timestamp(F.lit(f"{TARGET_Y:04d}-{TARGET_M:02d}-{TARGET_D:02d} {TARGET_H:02d}:00:00")).alias("t")
    ).collect()[0]["t"]
else:
    target_ts_val = df.select(F.max("ts_hour").alias("mx")).collect()[0]["mx"]

if target_ts_val is None:
    raise Exception("No data found in GOLD_V2")

hour_key       = hour_key_from_ts(target_ts_val)
SERVING_OUT_DIR= f"{SERVING_FEATURES_DIR}/hour_key={hour_key}"
print("Target hour:", target_ts_val, "| hour_key:", hour_key)

# Filter lookback window
start_ts_val = spark.sql(
    f"SELECT timestamp('{target_ts_val}') - INTERVAL {LOOKBACK_HOURS} HOURS AS st"
).collect()[0]["st"]

df_w = df.filter((F.col("ts_hour") >= F.lit(start_ts_val)) & (F.col("ts_hour") <= F.lit(target_ts_val)))
df_w = df_w.withColumn("date", F.to_date("ts_hour")).withColumn("dow_num", F.dayofweek("date"))

# Build lags + rolling (identical to original)
w = Window.partitionBy("station_id").orderBy(F.col("ts_hour"))

df_feat = (
    df_w
    .withColumn("lag1_dep",   F.lag("departures", 1).over(w))
    .withColumn("lag2_dep",   F.lag("departures", 2).over(w))
    .withColumn("lag24_dep",  F.lag("departures", 24).over(w))
    .withColumn("lag168_dep", F.lag("departures", 168).over(w))
    .withColumn("lag1_arr",   F.lag("arrivals", 1).over(w))
    .withColumn("lag2_arr",   F.lag("arrivals", 2).over(w))
    .withColumn("lag24_arr",  F.lag("arrivals", 24).over(w))
    .withColumn("lag168_arr", F.lag("arrivals", 168).over(w))
)

roll_w_3h  = w.rowsBetween(-3,  -1)
roll_w_24h = w.rowsBetween(-24, -1)

df_feat = (
    df_feat
    .withColumn("roll_mean_3h_dep",  F.avg("departures").over(roll_w_3h))
    .withColumn("roll_std_24h_dep",  F.stddev("departures").over(roll_w_24h))
    .withColumn("roll_mean_3h_arr",  F.avg("arrivals").over(roll_w_3h))
    .withColumn("roll_std_24h_arr",  F.stddev("arrivals").over(roll_w_24h))
)

required_cols = [
    "lag1_dep","lag2_dep","lag24_dep","lag168_dep",
    "lag1_arr","lag2_arr","lag24_arr","lag168_arr",
    "roll_mean_3h_dep","roll_std_24h_dep","roll_mean_3h_arr","roll_std_24h_arr",
    "temperature_2m_celsius","apparent_temperature_celsius",
    "event_active_nearby_flag","event_impact_score"
]

df_serv      = df_feat.filter(F.col("ts_hour") == F.lit(target_ts_val))
rows_raw     = df_serv.count()
df_serv      = df_serv.dropna(subset=required_cols)
rows_final   = df_serv.count()
df_serv      = df_serv.withColumn("processed_at_utc", F.current_timestamp())

print("Serving rows (raw):", rows_raw, "| after dropna:", rows_final)
df_serv.select("year","month","day","hour").distinct().show(truncate=False)

dbutils.fs.rm(SERVING_OUT_DIR, True)
df_serv.write.mode("overwrite").parquet(SERVING_OUT_DIR)
dbutils.fs.put(LATEST_PTR_PATH, hour_key, True)
print("Written to:", SERVING_OUT_DIR)
print("LATEST pointer updated:", LATEST_PTR_PATH, "->", hour_key)

### Validation Job A

In [0]:
BASE = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
print(dbutils.fs.ls(BASE))
print("LATEST:", dbutils.fs.head(f"{BASE}/LATEST.txt", 100))

## JOB B - Score Net Flow Hourly (LightGBM)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import zlib, json
import lightgbm as lgb

# ============================================================
# JOB B - Score Net Flow Hourly (Serverless-safe, contract-first)
# LightGBM version
# ============================================================

SERVING_FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
LATEST_PTR_PATH      = f"{SERVING_FEATURES_DIR}/LATEST.txt"
PREDICTIONS_DIR      = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/predictions/goldv2_netflow_hourly_lgbm"

WEIGHT_EVENT   = 10
DEP_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/dep_lgbm_weight{WEIGHT_EVENT}.txt"
ARR_MODEL_DBFS = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/arr_lgbm_weight{WEIGHT_EVENT}.txt"
META_DBFS      = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_lgbm/features_weight{WEIGHT_EVENT}/features.json"

MAX_MODEL_BYTES = 50_000_000
MAX_META_BYTES  = 500_000
BUCKET_COL      = "station_bucket"

def path_exists(path):
    try: dbutils.fs.ls(path); return True
    except: return False

def read_dbfs_text(p, mb): return dbutils.fs.head(p, mb)
def load_lgbm(p): return lgb.Booster(model_str=read_dbfs_text(p, MAX_MODEL_BYTES))
def station_to_bucket(sid, n): return zlib.crc32(str(sid).encode()) % n if sid else 0

def ensure_features(pdf, features, fill=0.0):
    for c in features:
        if c not in pdf.columns: pdf[c] = fill
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(fill)

def get_latest_hour_key() -> str:
    if path_exists(LATEST_PTR_PATH):
        hk = read_dbfs_text(LATEST_PTR_PATH, 100).strip()
        if hk: return hk
    keys = [it.name.rstrip("/").split("=",1)[1]
            for it in dbutils.fs.ls(SERVING_FEATURES_DIR)
            if it.name.rstrip("/").startswith("hour_key=")]
    if not keys: raise Exception("No hour_key folders and LATEST.txt missing.")
    return sorted(keys)[-1]

# Validate paths
for p in [SERVING_FEATURES_DIR, DEP_MODEL_DBFS, ARR_MODEL_DBFS, META_DBFS]:
    if not path_exists(p): raise Exception(f"Missing path: {p}")

# Load metadata contract
meta        = json.loads(read_dbfs_text(META_DBFS, MAX_META_BYTES))
DEP_FEATS   = meta["dep_features"]
ARR_FEATS   = meta["arr_features"]
HASH_BUCKETS= int(meta.get("hash_buckets", 512))

assert DEP_FEATS and ARR_FEATS, "Metadata missing dep/arr features"
assert len(set(DEP_FEATS)) == len(DEP_FEATS), "dep_features has duplicates"
assert len(set(ARR_FEATS)) == len(ARR_FEATS), "arr_features has duplicates"

KEY_COLS = ["station_id","year","month","day","hour","date","dow_num"]

# Get latest hour + read serving features
hour_key     = get_latest_hour_key()
SERVING_IN   = f"{SERVING_FEATURES_DIR}/hour_key={hour_key}"
PRED_OUT_DIR = f"{PREDICTIONS_DIR}/hour_key={hour_key}"

if not path_exists(SERVING_IN): raise Exception(f"Serving folder not found: {SERVING_IN}")
print("Scoring hour_key:", hour_key)

df_h = spark.read.parquet(SERVING_IN)
rows = df_h.count()
if rows == 0: raise Exception("No rows in serving features.")
print("Serving rows:", rows)

# Pandas feature prep
select_cols = list(dict.fromkeys(KEY_COLS + DEP_FEATS + ARR_FEATS))
pdf = df_h.select(*[c for c in select_cols if c in df_h.columns]).toPandas()
pdf["date"]     = pd.to_datetime(pdf["date"])
pdf[BUCKET_COL] = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
pdf["is_weekend"]= pdf["dow_num"].isin([1,7]).astype(np.int8)
ensure_features(pdf, DEP_FEATS)
ensure_features(pdf, ARR_FEATS)

X_dep = pdf[DEP_FEATS].astype(np.float32).values
X_arr = pdf[ARR_FEATS].astype(np.float32).values

# Load boosters + predict
dep_booster = load_lgbm(DEP_MODEL_DBFS)
arr_booster = load_lgbm(ARR_MODEL_DBFS)

dep_pred = dep_booster.predict(X_dep).astype(np.float32)
arr_pred = arr_booster.predict(X_arr).astype(np.float32)
net_pred = (arr_pred - dep_pred).astype(np.float32)

if np.isnan(net_pred).any(): raise Exception("NaNs in predictions!")

# Write predictions
out_pdf = pdf[["station_id","year","month","day","hour"]].copy()
out_pdf["dep_pred"]      = dep_pred
out_pdf["arr_pred"]      = arr_pred
out_pdf["net_pred"]      = net_pred
out_pdf["model_version"] = f"goldv2_lgbm_weight{WEIGHT_EVENT}"
out_pdf["scored_at_utc"] = pd.Timestamp.utcnow()

dbutils.fs.rm(PRED_OUT_DIR, True)
spark.createDataFrame(out_pdf).write.mode("overwrite").parquet(PRED_OUT_DIR)
print("Predictions written to:", PRED_OUT_DIR)

### Validation Job B

In [0]:
FEAT_BASE = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
PRED_BASE = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/predictions/goldv2_netflow_hourly_lgbm"
hk = dbutils.fs.head(f"{FEAT_BASE}/LATEST.txt", 100).strip()
print("hour_key:", hk)
print("pred folder:", dbutils.fs.ls(f"{PRED_BASE}/hour_key={hk}"))

## JOB C - Evaluate & Monitor Hourly NetFlow Predictions (LightGBM)

In [0]:
from pyspark.sql import functions as F
import json

# ============================================================
# JOB C - Evaluate & Monitor Hourly NetFlow Predictions
# Monitoring logic is identical to XGB version
# ============================================================

SERVING_FEATURES_DIR = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
LATEST_PTR_PATH      = f"{SERVING_FEATURES_DIR}/LATEST.txt"
PREDICTIONS_DIR      = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/predictions/goldv2_netflow_hourly_lgbm"
SCORED_DIR           = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/scored_with_actuals/goldv2_netflow_hourly_lgbm"
MONITORING_DIR       = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/monitoring/goldv2_netflow_hourly_lgbm"

WEIGHT_EVENT = 10
META_DBFS    = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_lgbm/features_weight{WEIGHT_EVENT}/features.json"

def path_exists(path):
    try: dbutils.fs.ls(path); return True
    except: return False

def get_latest_hour_key() -> str:
    if path_exists(LATEST_PTR_PATH):
        hk = dbutils.fs.head(LATEST_PTR_PATH, 100).strip()
        if hk: return hk
    keys = [it.name.rstrip("/").split("=",1)[1]
            for it in dbutils.fs.ls(SERVING_FEATURES_DIR)
            if it.name.rstrip("/").startswith("hour_key=")]
    if not keys: raise Exception("No hour_key folders found.")
    return sorted(keys)[-1]

for p in [SERVING_FEATURES_DIR, PREDICTIONS_DIR]:
    if not path_exists(p): raise Exception(f"Missing path: {p}")

hour_key      = get_latest_hour_key()
FEATURES_IN   = f"{SERVING_FEATURES_DIR}/hour_key={hour_key}"
PRED_IN       = f"{PREDICTIONS_DIR}/hour_key={hour_key}"

if not path_exists(FEATURES_IN): raise Exception(f"Features folder not found: {FEATURES_IN}")
if not path_exists(PRED_IN):     raise Exception(f"Predictions folder not found: {PRED_IN}")
print("Evaluating hour_key:", hour_key)

df_f = spark.read.parquet(FEATURES_IN)
df_p = spark.read.parquet(PRED_IN)

KEYS     = ["station_id","year","month","day","hour"]
need_f   = [c for c in KEYS + ["departures","arrivals","lag1_arr","lag1_dep","event_active_nearby_flag"] if c in df_f.columns]
need_p   = [c for c in KEYS + ["dep_pred","arr_pred","net_pred","scored_at_utc","model_version"] if c in df_p.columns]

df_j = df_f.select(need_f).join(df_p.select(need_p), on=KEYS, how="inner")
rows = df_j.count()
if rows == 0: raise Exception("No joined rows.")
print("Joined rows:", rows)

df_m = (
    df_j
    .withColumn("net_real",           (F.col("arrivals") - F.col("departures")).cast("double"))
    .withColumn("baseline_net",       (F.col("lag1_arr") - F.col("lag1_dep")).cast("double"))
    .withColumn("abs_err_net",        F.abs(F.col("net_pred").cast("double") - F.col("net_real")))
    .withColumn("sq_err_net",         F.pow(F.col("net_pred").cast("double") - F.col("net_real"), F.lit(2.0)))
    .withColumn("abs_err_baseline",   F.abs(F.col("baseline_net") - F.col("net_real")))
    .withColumn("sq_err_baseline",    F.pow(F.col("baseline_net") - F.col("net_real"), F.lit(2.0)))
    .withColumn("event_flag",         F.coalesce(F.col("event_active_nearby_flag").cast("int"), F.lit(0)))
    .withColumn("evaluated_at_utc",   F.current_timestamp())
)

# Write per-station scored output
SCORED_OUT = f"{SCORED_DIR}/hour_key={hour_key}"
dbutils.fs.rm(SCORED_OUT, True)
df_m.write.mode("overwrite").parquet(SCORED_OUT)
print("Scored written to:", SCORED_OUT)

# Global monitoring summary
agg_global = (
    df_m.agg(
        F.count("*").alias("rows"),
        F.avg("abs_err_net").alias("mae_net"),
        F.sqrt(F.avg("sq_err_net")).alias("rmse_net"),
        F.avg("abs_err_baseline").alias("mae_baseline_net"),
        F.sqrt(F.avg("sq_err_baseline")).alias("rmse_baseline_net"),
        F.avg("net_real").alias("avg_net_real"),
        F.avg("net_pred").alias("avg_net_pred"),
        F.max("abs_err_net").alias("max_abs_err_net"),
    )
    .withColumn("hour_key", F.lit(hour_key))
    .withColumn("improvement_pct_mae",
        F.when(F.col("mae_baseline_net") > 0,
               (F.col("mae_baseline_net") - F.col("mae_net")) / F.col("mae_baseline_net") * F.lit(100.0)
        ).otherwise(F.lit(None).cast("double")))
    .withColumn("evaluated_at_utc", F.current_timestamp())
)

# Segmented: event vs no-event
def seg_metrics(flag_val, prefix):
    d = df_m.filter(F.col("event_flag") == F.lit(flag_val))
    return d.agg(
        F.count("*").alias(f"{prefix}_rows"),
        F.avg("abs_err_net").alias(f"{prefix}_mae_net"),
        F.sqrt(F.avg("sq_err_net")).alias(f"{prefix}_rmse_net"),
        F.avg("abs_err_baseline").alias(f"{prefix}_mae_baseline_net"),
        F.sqrt(F.avg("sq_err_baseline")).alias(f"{prefix}_rmse_baseline_net"),
    )

monitor = agg_global.crossJoin(seg_metrics(1, "event")).crossJoin(seg_metrics(0, "no_event"))

MON_OUT = f"{MONITORING_DIR}/hour_key={hour_key}"
dbutils.fs.rm(MON_OUT, True)
monitor.write.mode("overwrite").parquet(MON_OUT)
print("Monitoring written to:", MON_OUT)
display(monitor)

### Validation Job C

In [0]:
FEAT_BASE = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
hk = dbutils.fs.head(f"{FEAT_BASE}/LATEST.txt", 100).strip()
SCORED = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/scored_with_actuals/goldv2_netflow_hourly_lgbm/hour_key={hk}"
MON    = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/monitoring/goldv2_netflow_hourly_lgbm/hour_key={hk}"
print("scored:",     dbutils.fs.ls(SCORED))
print("monitoring:", dbutils.fs.ls(MON))

## JOB D - Validation Final (LightGBM)

In [0]:
from pyspark.sql import functions as F
import pandas as pd
import numpy as np
import zlib, json
import lightgbm as lgb

# ============================================================
# JOB D - VALIDATION / CONTRACT TEST (A->B->C artifacts)
# LightGBM version - Serverless-safe
# ============================================================

WEIGHT_EVENT          = 10
SERVING_FEATURES_BASE = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/features/goldv2_hourly_features_lgbm"
PREDICTIONS_BASE      = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/predictions/goldv2_netflow_hourly_lgbm"
MONITORING_BASE       = "dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/monitoring/goldv2_netflow_hourly_lgbm"
DEP_MODEL_DBFS        = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/dep_lgbm_weight{WEIGHT_EVENT}.txt"
ARR_MODEL_DBFS        = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/models_lgbm/arr_lgbm_weight{WEIGHT_EVENT}.txt"
META_DBFS             = f"dbfs:/Volumes/workspace/default/dbfs/Projects/Capstone/serving/metadata_lgbm/features_weight{WEIGHT_EVENT}/features.json"
LATEST_PTR            = f"{SERVING_FEATURES_BASE}/LATEST.txt"
BUCKET_COL            = "station_bucket"

def path_exists(p):
    try: dbutils.fs.ls(p); return True
    except: return False

def read_text(p, mb): return dbutils.fs.head(p, mb)
def load_lgbm(p): return lgb.Booster(model_str=read_text(p, 50_000_000))
def station_to_bucket(sid, n): return zlib.crc32(str(sid).encode()) % n if sid else 0

def ensure_features(pdf, feats, fill=0.0):
    for c in feats:
        if c not in pdf.columns: pdf[c] = fill
        pdf[c] = pd.to_numeric(pdf[c], errors="coerce").fillna(fill)

def assert_true(cond, msg):
    if not cond: raise Exception(f"VALIDATION FAILED: {msg}")

# 1) Artifact existence
for p in [DEP_MODEL_DBFS, ARR_MODEL_DBFS, META_DBFS, SERVING_FEATURES_BASE, LATEST_PTR]:
    assert_true(path_exists(p), f"Missing: {p}")
print("DBFS paths exist")

# 2) Metadata contract
meta          = json.loads(read_text(META_DBFS, 500_000))
DEP_FEATS     = meta["dep_features"]
ARR_FEATS     = meta["arr_features"]
HASH_BUCKETS  = int(meta.get("hash_buckets", 512))
EVENT_FLAG_COL= meta.get("event_flag_col", "event_active_nearby_flag")
MODEL_TYPE    = meta.get("model_type", "?")

assert_true(len(DEP_FEATS) > 0, "dep_features empty")
assert_true(len(ARR_FEATS) > 0, "arr_features empty")
print(f"Metadata OK. model_type={MODEL_TYPE}, dep={len(DEP_FEATS)}, arr={len(ARR_FEATS)}, hash_buckets={HASH_BUCKETS}")

# 3) Load boosters
dep_booster = load_lgbm(DEP_MODEL_DBFS)
arr_booster = load_lgbm(ARR_MODEL_DBFS)
assert_true(dep_booster.num_trees() > 0, "dep booster has 0 trees")
assert_true(arr_booster.num_trees() > 0, "arr booster has 0 trees")
print(f"Boosters OK. dep_trees={dep_booster.num_trees()}, arr_trees={arr_booster.num_trees()}")

# 4) Latest hour_key
latest_key = read_text(LATEST_PTR, 100).strip()
assert_true(len(latest_key) == 10 and latest_key.isdigit(), f"LATEST.txt malformed: '{latest_key}'")

FEATURES_DIR = f"{SERVING_FEATURES_BASE}/hour_key={latest_key}"
PRED_DIR     = f"{PREDICTIONS_BASE}/hour_key={latest_key}"
MON_DIR      = f"{MONITORING_BASE}/hour_key={latest_key}"

assert_true(path_exists(FEATURES_DIR), f"Missing features folder: {FEATURES_DIR}")
print(f"Latest hour_key: {latest_key}")

# 5) Load features + build X
df_feat  = spark.read.parquet(FEATURES_DIR)
KEY_COLS = ["station_id","year","month","day","hour","date","dow_num"]
for c in KEY_COLS: assert_true(c in df_feat.columns, f"Missing key col: {c}")

rows = df_feat.count()
assert_true(rows > 0, "No rows in serving features")
print("Serving rows:", rows)

select_cols = list(dict.fromkeys(KEY_COLS + DEP_FEATS + ARR_FEATS))
pdf = df_feat.select(*[c for c in select_cols if c in df_feat.columns]).toPandas()
pdf["date"]      = pd.to_datetime(pdf["date"])
pdf[BUCKET_COL]  = pdf["station_id"].map(lambda s: station_to_bucket(s, HASH_BUCKETS)).astype(np.int16)
pdf["is_weekend"]= pdf["dow_num"].isin([1,7]).astype(np.int8)
ensure_features(pdf, DEP_FEATS)
ensure_features(pdf, ARR_FEATS)

X_dep = pdf[DEP_FEATS].astype(np.float32).values
X_arr = pdf[ARR_FEATS].astype(np.float32).values
assert_true(X_dep.shape == (rows, len(DEP_FEATS)), "X_dep shape mismatch")
assert_true(X_arr.shape == (rows, len(ARR_FEATS)), "X_arr shape mismatch")
print("Feature shapes OK:", X_dep.shape, X_arr.shape)

# 6) Predict sanity
dep_pred = dep_booster.predict(X_dep).astype(np.float32)
arr_pred = arr_booster.predict(X_arr).astype(np.float32)
net_pred = (arr_pred - dep_pred).astype(np.float32)

assert_true(not np.isnan(dep_pred).any(), "dep_pred NaNs")
assert_true(not np.isnan(arr_pred).any(), "arr_pred NaNs")
assert_true(not np.isnan(net_pred).any(), "net_pred NaNs")
print("Predictions OK (no NaNs)")
print(f"dep_pred: min={dep_pred.min():.2f} mean={dep_pred.mean():.2f} max={dep_pred.max():.2f}")
print(f"arr_pred: min={arr_pred.min():.2f} mean={arr_pred.mean():.2f} max={arr_pred.max():.2f}")
print(f"net_pred: min={net_pred.min():.2f} mean={net_pred.mean():.2f} max={net_pred.max():.2f}")

# 7) Job B output
assert_true(path_exists(PRED_DIR), f"Predictions folder missing (run Job B): {PRED_DIR}")
df_pred   = spark.read.parquet(PRED_DIR)
pred_rows = df_pred.count()
assert_true(pred_rows > 0, "No rows in predictions")
for c in ["station_id","year","month","day","hour","dep_pred","arr_pred","net_pred"]:
    assert_true(c in df_pred.columns, f"Predictions missing column: {c}")
print("Predictions folder OK. rows:", pred_rows)

# 8) Job C output
assert_true(path_exists(MON_DIR), f"Monitoring folder missing (run Job C): {MON_DIR}")
df_mon   = spark.read.parquet(MON_DIR)
mon_rows = df_mon.count()
assert_true(mon_rows > 0, "Monitoring summary empty")
print("Monitoring folder OK. rows:", mon_rows)

print("\nALL VALIDATIONS PASSED: LGBM artifacts + contract + A/B/C outputs are consistent.")

**Output Validation**

The generated feature set is validated to ensure compatibility with the trained models.

This includes schema consistency and data completeness checks.